# Big Data Hypothesis Analysis — Amazon Electronics Reviews (Q2 2014)

**Module:** 6G6Z0030 High Performance Computing & Big Data  
**Student:** Challa Harikrishna Nagasai Charan  
**MMU ID:** 23727842

This notebook tests the research hypothesis: *"In the second quarter of 2014, products given a review rating of 3 or more are significantly different compared to other products."* The Amazon Electronics Reviews dataset from Kaggle is loaded into Apache Spark, filtered to Q2 2014, and analysed at both the group level and the per-product level (to avoid circular reasoning).

**Note on duplicate setup cells:** This notebook was developed across multiple Google Colab sessions. Some Spark setup cells appear more than once because Colab session timeouts required reinstalling Spark and reconfiguring the environment in fresh runtimes. Only the final, successful setup chain is preserved below.

## 1. Environment setup

Apache Spark 3.1.2 with Hadoop 2.7 was installed on the Google Colab runtime. Spark 3.1.2 was used because it is compatible with the Java 8 toolchain available in Colab and matches the version used during course laboratory work. `findspark` is used to register Spark with the Python environment.

In [1]:
# Remove broken download
!rm -rf spark-3.1.2-bin-hadoop2.7*
!rm -f spark-3.1.2-bin-hadoop2.7.tgz

# Re-download properly
!wget https://archive.apache.org/dist/spark/spark-3.1.2/spark-3.1.2-bin-hadoop2.7.tgz

# Extract
!tar -xvzf spark-3.1.2-bin-hadoop2.7.tgz

# Install findspark
!pip install -q findspark

--2026-05-18 20:20:20--  https://archive.apache.org/dist/spark/spark-3.1.2/spark-3.1.2-bin-hadoop2.7.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 224445805 (214M) [application/x-gzip]
Saving to: ‘spark-3.1.2-bin-hadoop2.7.tgz’

spark-3.1.2-bin-had 100%[===================>] 214.05M   424KB/s    in 7m 20s  

2026-05-18 20:27:40 (498 KB/s) - ‘spark-3.1.2-bin-hadoop2.7.tgz’ saved [224445805/224445805]

spark-3.1.2-bin-hadoop2.7/
spark-3.1.2-bin-hadoop2.7/R/
spark-3.1.2-bin-hadoop2.7/R/lib/
spark-3.1.2-bin-hadoop2.7/R/lib/sparkr.zip
spark-3.1.2-bin-hadoop2.7/R/lib/SparkR/
spark-3.1.2-bin-hadoop2.7/R/lib/SparkR/worker/
spark-3.1.2-bin-hadoop2.7/R/lib/SparkR/worker/worker.R
spark-3.1.2-bin-hadoop2.7/R/lib/SparkR/worker/daemon.R
spark-3.1.2-bin-hadoop2.7/R/lib/SparkR/tests/
spark-3.1.2-bin-hadoop2.7/R/lib

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.2-bin-hadoop2.7"

import findspark
findspark.init()

print("Spark configured")

Spark configured


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonBigData") \
    .getOrCreate()

print("Spark is running 🚀")

Spark is running 🚀


## 2. Upload and load the dataset

The Amazon Electronics Reviews dataset (`ratings_Electronics.csv`) from Kaggle is uploaded to the Colab runtime and loaded as a Spark DataFrame. The four columns are renamed to `userId`, `productId`, `rating` and `timestamp`. The first five rows are displayed to confirm successful loading.

In [9]:
from google.colab import files
uploaded = files.upload()

Saving ratings_Electronics (1).csv to ratings_Electronics (1).csv


In [10]:
df = spark.read.csv('/content/ratings_Electronics (1).csv', header=False)

df = df.toDF("userId", "productId", "rating", "timestamp")

df.show(5)

+--------------+----------+------+----------+
|        userId| productId|rating| timestamp|
+--------------+----------+------+----------+
| AKM1MP6P0OYPR|0132793040|   5.0|1365811200|
|A2CX7LUOHB2NDG|0321732944|   5.0|1341100800|
|A2NWSAGRHCP8N5|0439886341|   1.0|1367193600|
|A2WNBOD3WNDNKT|0439886341|   3.0|1374451200|
|A1GI0U4ZRJA8WN|0439886341|   1.0|1334707200|
+--------------+----------+------+----------+
only showing top 5 rows



## 3. Cast types and convert timestamp

The `rating` column is cast to `float` and the `timestamp` column (Unix epoch seconds) is cast to `long` and then converted into a human-readable `date` column for filtering.

In [11]:
from pyspark.sql.functions import col

df = df.withColumn("rating", col("rating").cast("float"))
df = df.withColumn("timestamp", col("timestamp").cast("long"))

In [12]:
from pyspark.sql.functions import from_unixtime

df = df.withColumn("date", from_unixtime(col("timestamp")))

## 4. Filter to Q2 2014 (April–June)

Reviews are filtered to retain only those posted in April, May or June 2014 (the second quarter). The total record count is printed.

In [13]:
from pyspark.sql.functions import year, month

q2_2014 = df.filter(
    (year("date") == 2014) &
    (month("date").isin(4, 5, 6))
)

print("Q2 count:", q2_2014.count())

Q2 count: 664014


## 5. Group-level analysis (rating-based comparison)

Reviews are split into two groups: those with `rating ≥ 3` (high) and those with `rating < 3` (low). The mean rating and count of each group are computed.

**Note:** comparing the *mean rating* between groups defined by rating thresholds is circular by construction. The per-product analysis in the next section addresses this by comparing a different attribute (per-product review counts) across the two groups.

In [14]:
high_ratings = q2_2014.filter(col("rating") >= 3)
low_ratings = q2_2014.filter(col("rating") < 3)

In [15]:
high_avg = high_ratings.groupBy().avg("rating").collect()[0][0]
low_avg = low_ratings.groupBy().avg("rating").collect()[0][0]

print("High avg:", high_avg)
print("Low avg:", low_avg)

High avg: 4.603885158172088
Low avg: 1.3270304624150777


In [16]:
print("High count:", high_ratings.count())
print("Low count:", low_ratings.count())

High count: 549939
Low count: 114075


## 6. Per-product analysis (non-circular comparison)

To avoid the circular reasoning of the previous section, products are aggregated by `productId` and characterised by their **mean rating** and **review count**. Products are then classified into high-rated (mean ≥ 3) and low-rated (mean < 3) groups. The two groups are compared on a *different* attribute — the average number of reviews per product — which gives a genuine, non-circular test of whether the two populations differ in a meaningful characteristic beyond their ratings.

**Expected interpretation:** If high-rated products receive substantially more reviews per product than low-rated products, this supports the hypothesis that the two groups differ in user engagement, not only in rating values.

In [17]:
from pyspark.sql.functions import avg, count, col

products = q2_2014.groupBy("productId").agg(
    avg("rating").alias("mean_rating"),
    count("rating").alias("num_reviews")
)

high_products = products.filter(col("mean_rating") >= 3)
low_products  = products.filter(col("mean_rating") < 3)

high_reviews_per_product = high_products.agg(avg("num_reviews")).collect()[0][0]
low_reviews_per_product  = low_products.agg(avg("num_reviews")).collect()[0][0]

print("High products:", high_products.count())
print("Low products:", low_products.count())
print("Avg reviews (high):", high_reviews_per_product)
print("Avg reviews (low):", low_reviews_per_product)

High products: 102116
Low products: 19900
Avg reviews (high): 5.978230639664695
Avg reviews (low): 2.690502512562814


## 6.2 Formal statistical test (Mann-Whitney U)

The per-product review-count distributions are heavily right-skewed (a small number of products receive a very large number of reviews), so a non-parametric Mann-Whitney U test is more appropriate than a t-test. The Spark DataFrames are converted to local numpy arrays (the per-product table is small enough to fit in memory: ~122,000 rows total) and `scipy.stats.mannwhitneyu` is used with a two-sided alternative. The reported p-value supports the descriptive 2.2× ratio finding.

In [18]:
from scipy.stats import mannwhitneyu
import numpy as np

# Collect per-product review counts into local arrays.
# The per-product table has ~122k rows total and fits comfortably in memory.
high_counts = np.array([r['num_reviews'] for r in high_products.select('num_reviews').collect()])
low_counts  = np.array([r['num_reviews'] for r in low_products.select('num_reviews').collect()])

print('High n:', len(high_counts), 'Low n:', len(low_counts))
print('High median:', float(np.median(high_counts)), 'Low median:', float(np.median(low_counts)))

u_stat, p_value = mannwhitneyu(high_counts, low_counts, alternative='two-sided')
print(f'Mann-Whitney U statistic: {u_stat:.3e}')
print(f'p-value: {p_value:.3e}')

if p_value < 0.001:
    print('Result: difference is statistically significant at p < 0.001')
elif p_value < 0.05:
    print('Result: difference is statistically significant at p < 0.05')
else:
    print('Result: difference is NOT statistically significant')


High n: 102116 Low n: 19900
High median: 1.0 Low median: 1.0
Mann-Whitney U statistic: 1.187e+09
p-value: 0.000e+00
Result: difference is statistically significant at p < 0.001


## 7. Summary of findings

The analysis returned the following results for Q2 2014:

| Metric | High-rated (≥3) | Low-rated (<3) |
|---|---|---|
| Mean rating | 4.6039 | 1.3270 |
| Review count | 549,939 | 114,075 |
| Distinct products | 102,116 | 19,900 |
| Avg reviews per product | 5.98 | 2.69 |

**Key finding:** high-rated products received approximately 2.2× more reviews per product on average (5.98 vs 2.69). This is a non-circular comparison and supports the research hypothesis that high-rated and low-rated products differ in user engagement.

**Formal test:** the Mann-Whitney U test in Section 6.2 confirms the per-product review-count distributions differ significantly between the two groups. Given the sample sizes (102,116 vs 19,900) the test has very high statistical power, so the practically meaningful finding remains the 2.2× ratio, not the p-value in isolation.

**Caveat:** the analysis is descriptive and does not establish causation. It cannot distinguish whether being highly rated causes more reviews, whether being heavily reviewed drives higher ratings, or whether both are driven by an unobserved factor such as product quality or marketing spend.